# 归档材料

本节加载模型后汇总旧 artifact；它没有用模型输出重新驱动规划、控制和评测，因此不构成端到端交付。

[归档索引](../README.md) · [当前学习入口](../../../course/first_loop/README.md)

# 10 · 归档审计：模型与报告之间还缺什么连接？

本节检查旧章节生成的文件，并运行一次 BEV 模型推理。下图区分实际执行和原设计中尚未执行的连接：

```text
实际执行：05_bev_model.pt → BEV risk/occupancy inference
实际执行：读取 03/06/07/08/09 的旧文件 → 汇总表

原设计设想（本节未执行）：
BEV inference --未连接--> prediction --未连接--> ego planner
ego planner --未连接--> env.step --未连接--> 新的闭环评测
```

审计练习：找出实际推理调用和汇总表的输入。若要判断换模型后车是否开得更好，还需要在哪个函数中把模型输出送入规划，再调用 `env.step`？现有执行结果只支持文件可读取和推理可运行。

In [ ]:
from pathlib import Path
import sys

PROJECT_ROOT = next(path for path in (Path.cwd(), *Path.cwd().parents)
                    if (path / "src" / "ad_tutorial").is_dir())
sys.path.insert(0, str(PROJECT_ROOT / "src"))

from ad_tutorial import (
    ARTIFACT_DIR,
    BEVConfig,
    build_bev_dataset,
    build_urban_cut_in_scene,
    ensure_artifact_dir,
    load_json_artifact,
    load_numpy_artifact,
    save_json_artifact,
    save_numpy_artifact,
    scene_to_bev,
)

ensure_artifact_dir()
print("project root:", PROJECT_ROOT)
print("artifact directory:", ARTIFACT_DIR)

import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

required = [
    "00_system_contract.json", "01_geometry.json", "02_bev_dataset.npz",
    "03_temporal_state.npz", "04_localization.npz", "05_bev_model.pt",
    "06_prediction.npz", "07_planner.npz", "08_eval_report.json", "09_safety_runtime.json",
]
artifact_status = {name: (ARTIFACT_DIR / name).exists() for name in required}
print(artifact_status)
missing = [name for name, present in artifact_status.items() if not present]
if missing:
    raise FileNotFoundError("Run Chapters 00–09 in order; missing artifacts: " + ", ".join(missing))

## 1. Load the learned BEV artifact and run one inference

The model class is shared with Chapter 05 so this is not a prose-only reference to a checkpoint. We load the exact state dict, run one sample from the same BEV artifact, and expose a risk summary to the downstream report. The planner remains a separate component; it should not silently become a learned model just because a checkpoint exists.

In [ ]:
import torch
from ad_tutorial.bev_model import LearnableBEVModel

bev_data = load_numpy_artifact("02_bev_dataset.npz")
checkpoint = torch.load(ARTIFACT_DIR / "05_bev_model.pt", map_location="cpu", weights_only=False)
config = checkpoint["model_config"]
bev_model = LearnableBEVModel(**config)
bev_model.load_state_dict(checkpoint["state_dict"])
bev_model.eval()
with torch.no_grad():
    logits = bev_model(torch.tensor(bev_data["features"][:1], dtype=torch.float32))
    risk_probability = torch.sigmoid(logits[0, 1]).numpy()
print("loaded BEV checkpoint:", config)
print("predicted risk cells:", int((risk_probability > 0.5).sum()), "max risk:", float(risk_probability.max()))

In [ ]:
prediction = load_numpy_artifact("06_prediction.npz")
planner = load_numpy_artifact("07_planner.npz")
eval_report = load_json_artifact("08_eval_report.json")
safety = load_json_artifact("09_safety_runtime.json")
temporal = load_numpy_artifact("03_temporal_state.npz")

summary = pd.DataFrame([
    {"stage": "BEV model", "artifact": "05_bev_model.pt", "evidence": f"risk cells={int((risk_probability > 0.5).sum())}"},
    {"stage": "Prediction", "artifact": "06_prediction.npz", "evidence": f"modes={prediction['candidates'].shape[0]}, minFDE={prediction['per_mode_fde'].min():.2f}m"},
    {"stage": "Planner", "artifact": "07_planner.npz", "evidence": f"selected_mode={int(planner['selected_mode'][0])}"},
    {"stage": "Evaluation", "artifact": "08_eval_report.json", "evidence": f"collision_rate={eval_report['collision_rate']:.3f}"},
    {"stage": "Safety/runtime", "artifact": "09_safety_runtime.json", "evidence": str(safety['release_checks'])},
])
display(summary)

In [ ]:
# One compact failure replay: stale sensor input + cut-in risk.
failure = {
    "scenario_id": "urban_cut_in_failure_replay",
    "root_cause_hypothesis": "sensor age and cut-in mode were not reflected in nominal policy",
    "observed_signals": {"risk_cells": int((risk_probability > 0.5).sum()),
                         "min_prediction_fde_m": float(prediction["per_mode_fde"].min()),
                         "eval_collision_rate": float(eval_report["collision_rate"])},
    "safety_action": "DEGRADED or MINIMAL_RISK depending on TTC/latency",
    "next_experiment": "fixed nuScenes mini replay with the same bundle/slice contract",
}
print(json.dumps(failure, indent=2))
fig, ax = plt.subplots(figsize=(8, 4))
ax.imshow(risk_probability.T, origin="lower", aspect="auto", cmap="magma")
ax.set(title="Loaded BEV risk map from the trained artifact", xlabel="x cell", ylabel="y cell")
plt.show()

## Portfolio handoff

Commit these items together:

- exact environment/requirements and command;
- data/scene version, coordinate and timestamp policy;
- model checkpoint metadata, seed and training split;
- open-loop metrics, closed-loop metrics, safety gate, p50/p95/p99 latency;
- at least two ablations and one failure replay;
- explicit boundary: synthetic mechanism tutorial ≠ public benchmark ≠ vehicle safety case.

Optional labs now sit after this core: Flow Matching/Action Chunk, VLM structured conditions, and VLA/WA/π0. They extend interfaces already built here; they do not replace the core perception–state–planning–safety chain.